# 08 — Full 20-adapter test analysis

**Purpose.** Analyze notebooks 07 and 07.1 without running the model. Headline metrics
exclude literal own-secret leaks and use the primary global emitted-token-ID
mask. The validation-frozen layer/position are confirmatory anchors; scans over
all test layers and positions are explicitly exploratory.

The notebook reproduces Logit-Lens-style top-1/top-5 Accuracy, Pass@10 and
Majority@10 for both LL and J-Lens, reports robust rank/probability summaries,
audits masking sensitivity, maps layers × understandable token positions, retains
decoded internal top-token examples, and compares matching adapters against the
paired base-model control for all 20 candidate words.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import re
import sys
import time
import unicodedata
from collections import Counter, defaultdict
from importlib.metadata import distribution
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


## Open the completed run and verify all atomic files


In [ ]:
from src.experiment_io import load_json, open_run, utc_now

analysis_pointer = PROJECT_ROOT / "results" / "latest_qwen36_20_adapter_test_run.json"
assert analysis_pointer.exists(), "Run notebook 07 first."
analysis_pointer_data = load_json(analysis_pointer)
ANALYSIS_RUN_ID = os.environ.get("QWEN_TEST_RUN_ID", analysis_pointer_data["run_id"])
analysis_paths, analysis_config = open_run(ANALYSIS_RUN_ID)
completion = load_json(analysis_paths.result_dir / "test_sweep_completion.json")
assert completion["completed_sequences"] == completion["expected_sequences"] == 4000

analysis_cells_dir = analysis_paths.lens_dir / "test_cells"
analysis_done_files = sorted(analysis_cells_dir.glob("*.done.json"))
analysis_aggregate_files = sorted(analysis_cells_dir.glob("*.aggregate.parquet"))
analysis_position_files = sorted(analysis_cells_dir.glob("*.positions.parquet"))
assert len(analysis_done_files) == len(analysis_aggregate_files) == len(analysis_position_files) == 4000

base_analysis_pointer = PROJECT_ROOT / "results/latest_qwen36_base_test_control_run.json"
assert base_analysis_pointer.exists(), "Run completed notebook 07.1 first."
base_analysis_pointer_data = load_json(base_analysis_pointer)
BASE_ANALYSIS_RUN_ID = os.environ.get(
    "QWEN_BASE_TEST_RUN_ID", base_analysis_pointer_data["run_id"]
)
base_analysis_paths, base_analysis_config = open_run(BASE_ANALYSIS_RUN_ID)
base_completion = load_json(
    base_analysis_paths.result_dir / "base_test_sweep_completion.json"
)
assert base_completion["completed_sequences"] == base_completion["expected_sequences"] == 200
assert base_completion["source_adapter_test_run_id"] == ANALYSIS_RUN_ID
base_analysis_cells_dir = base_analysis_paths.lens_dir / "base_test_cells"
base_analysis_done_files = sorted(base_analysis_cells_dir.glob("*.done.json"))
base_analysis_aggregate_files = sorted(base_analysis_cells_dir.glob("*.aggregate.parquet"))
base_analysis_position_files = sorted(base_analysis_cells_dir.glob("*.positions.parquet"))
assert len(base_analysis_done_files) == len(base_analysis_aggregate_files) == len(base_analysis_position_files) == 200

print("adapter run:", ANALYSIS_RUN_ID)
print("base control run:", BASE_ANALYSIS_RUN_ID)
print("adapter aggregate GiB:", round(sum(p.stat().st_size for p in analysis_aggregate_files) / 2**30, 3))
print("adapter positions GiB:", round(sum(p.stat().st_size for p in analysis_position_files) / 2**30, 3))
print("base aggregate GiB:", round(sum(p.stat().st_size for p in base_analysis_aggregate_files) / 2**30, 3))
print("base positions GiB:", round(sum(p.stat().st_size for p in base_analysis_position_files) / 2**30, 3))
display(completion)
display(base_completion)


## Load behavior and response-average readouts

One aggregate row is one `prompt × adapter × layer × method × mask protocol`.
`target_rank=1` is best. MRR is the mean of `1/rank`, so it lies in `[0,1]`
and is much less dominated by rare very large ranks than mean rank.


In [ ]:
from src.experiment_io import read_jsonl

behavior_path = analysis_paths.raw_dir / "test_behavior_generations.jsonl"
behavior = pd.DataFrame(read_jsonl(behavior_path)).drop_duplicates(
    ["prompt_id", "condition"], keep="last"
)
assert len(behavior) == 4000
leak_keys = set(
    zip(
        behavior.loc[behavior["own_secret_leaked"], "prompt_id"],
        behavior.loc[behavior["own_secret_leaked"], "condition"],
    )
)
print("literal own-secret leaks excluded from headlines:", len(leak_keys))

aggregate = pd.concat(
    [pd.read_parquet(path) for path in analysis_aggregate_files],
    ignore_index=True,
)
assert set(aggregate["split"]) == {"test"}
assert set(aggregate["method"]) == {"logit_lens", "jlens"}
assert set(aggregate["mask_protocol"]) == {
    "global_emitted_ids", "position_actual_token", "unmasked"
}
valid_aggregate = aggregate[~aggregate["own_secret_leaked"]].copy()
primary_aggregate = valid_aggregate[
    valid_aggregate["mask_protocol"].eq(
        analysis_config["readout"]["primary_mask_protocol"]
    )
].copy()

base_behavior_path = base_analysis_paths.raw_dir / "base_test_behavior_generations.jsonl"
base_behavior = pd.DataFrame(read_jsonl(base_behavior_path)).drop_duplicates(
    ["prompt_id"], keep="last"
)
assert len(base_behavior) == 200
assert set(base_behavior["condition"]) == {"base"}
assert set(base_behavior["prompt_id"]) == set(behavior["prompt_id"])
base_aggregate = pd.concat(
    [pd.read_parquet(path) for path in base_analysis_aggregate_files],
    ignore_index=True,
)
assert set(base_aggregate["split"]) == {"test"}
assert set(base_aggregate["condition"]) == {"base"}
assert set(base_aggregate["method"]) == {"logit_lens", "jlens"}
assert set(base_aggregate["mask_protocol"]) == {
    "global_emitted_ids", "position_actual_token", "unmasked"
}
assert set(base_aggregate["source_adapter_test_run_id"]) == {ANALYSIS_RUN_ID}

print("adapter aggregate rows:", len(aggregate), "primary valid rows:", len(primary_aggregate))
print("base aggregate rows:", len(base_aggregate))
display(
    behavior.groupby(["prompt_type", "condition"], as_index=False)
    .agg(
        sequences=("prompt_id", "size"),
        leaks=("own_secret_leaked", "sum"),
        mean_generation_tokens=("generation_token_count", "mean"),
    )
    .head(20)
)
display(
    base_behavior.groupby("prompt_type", as_index=False).agg(
        sequences=("prompt_id", "size"),
        mean_generation_tokens=("generation_token_count", "mean"),
        responses_mentioning_any_candidate=("any_candidate_mentioned", "sum"),
    )
)


## Robust rank and probability metrics across layers

The table avoids relying on mean rank alone:

- **median rank**: typical vocabulary position (lower is better);
- **geometric mean rank**: exponentiated mean log-rank, stable under rank jumps;
- **MRR**: strongly rewards ranks near 1 (higher is better);
- **Hit@k**: fraction with rank at most `k`;
- **candidate rank/share**: where the secret sits among only the 20 taboo words;
- **NLL / probability mass**: strength assigned to all audited one-token forms.


In [ ]:
def geometric_mean_rank(series):
    clean = series.dropna().astype(float)
    return float(np.exp(np.log(clean).mean())) if len(clean) else np.nan

def q25(series):
    return series.quantile(0.25)

def q75(series):
    return series.quantile(0.75)

rank_by_layer = (
    primary_aggregate.groupby(["prompt_type", "method", "layer"], as_index=False)
    .agg(
        sequences=("prompt_id", "size"),
        median_rank=("target_rank", "median"),
        rank_q25=("target_rank", q25),
        rank_q75=("target_rank", q75),
        geometric_mean_rank=("target_rank", geometric_mean_rank),
        mrr=("target_reciprocal_rank", "mean"),
        hit_at_1=("target_hit_top1", "mean"),
        hit_at_5=("target_hit_top5", "mean"),
        hit_at_10=("target_hit_top10", "mean"),
        hit_at_100=("target_hit_top100", "mean"),
        median_candidate_rank_20=("target_candidate_rank_20", "median"),
        mean_candidate_probability_share=("target_candidate_probability_share", "mean"),
        mean_target_probability_mass=("target_probability_mass", "mean"),
        mean_target_nll=("target_negative_log_probability", "mean"),
    )
)
rank_by_layer.to_csv(
    analysis_paths.result_dir / "test_rank_probability_metrics_by_layer.csv", index=False
)

anchor_layers = sorted({
    analysis_config["paper_reference"]["reported_layer"],
    analysis_config["validation_anchor"]["shared_response_layer"],
})
anchor_rank_table = rank_by_layer[
    rank_by_layer["layer"].isin(anchor_layers)
].sort_values(["prompt_type", "layer", "method"])
anchor_rank_table.to_csv(
    analysis_paths.result_dir / "test_anchor_layer_rank_table.csv", index=False
)
display(anchor_rank_table)


## Paper-style Accuracy, Pass@10 and Majority@10

For every adapter and fixed source-order block of ten prompts:

- **Accuracy** is the fraction of individual responses whose response-averaged
  distribution contains the secret in top-k.
- **Pass@10** is 1 when at least one of the ten responses succeeds.
- **Majority@10** frequency-votes the ten top-k lists; all one-token forms of
  the secret are canonicalized to one `TARGET` candidate.

We report both mask interpretations. Only `global_emitted_ids` is primary for
this project. The paper's Gemma-2-9B values are context, not a like-for-like
Qwen benchmark.


In [ ]:
from math import sqrt

def wilson_interval(successes, trials, z=1.96):
    if trials == 0:
        return (np.nan, np.nan)
    p = successes / trials
    denominator = 1 + z * z / trials
    center = (p + z * z / (2 * trials)) / denominator
    radius = z * sqrt(
        p * (1 - p) / trials + z * z / (4 * trials * trials)
    ) / denominator
    return (max(0.0, center - radius), min(1.0, center + radius))

def parse_ids(value):
    return [int(item) for item in json.loads(value)]

def majority_success(block, k):
    counts = Counter()
    probability_sums = defaultdict(float)
    target_ids = set(parse_ids(block.iloc[0]["target_token_ids_json"]))
    for row in block.to_dict("records"):
        seen = set()
        for item in json.loads(row["top10_json"])[:k]:
            token_id = int(item["token_id"])
            candidate = "TARGET" if token_id in target_ids else f"token:{token_id}"
            if candidate in seen:
                continue
            seen.add(candidate)
            counts[candidate] += 1
            probability_sums[candidate] += float(item["probability"])
    winners = sorted(
        counts,
        key=lambda item: (-counts[item], -probability_sums[item], item),
    )[:k]
    return "TARGET" in winners, winners

def build_paper_metrics(frame):
    unit_rows = []
    grouping = [
        "mask_protocol", "prompt_type", "condition",
        "paper_block_of_10", "method", "layer",
    ]
    for keys, block in frame.groupby(grouping, sort=True):
        mask_protocol, prompt_type, condition, block_id, method, layer = keys
        for k in analysis_config["readout"]["top_ks"]:
            hit_column = f"target_hit_top{k}"
            majority, winners = majority_success(block, k)
            unit_rows.append({
                "mask_protocol": mask_protocol,
                "prompt_type": prompt_type,
                "condition": condition,
                "paper_block_of_10": int(block_id),
                "method": method,
                "layer": int(layer),
                "top_k": int(k),
                "attempts": len(block),
                "eligible_at_10": len(block) == analysis_config["prompts"]["paper_block_size"],
                "accuracy": float(block[hit_column].astype(bool).mean()),
                "pass_at_10": bool(block[hit_column].astype(bool).any()),
                "majority_at_10": bool(majority),
                "majority_winners_json": json.dumps(winners),
            })
    units = pd.DataFrame(unit_rows)

    metric_rows = []
    for keys, group in units.groupby(
        ["mask_protocol", "prompt_type", "method", "layer", "top_k"]
    ):
        mask_protocol, prompt_type, method, layer, k = keys
        accuracy_trials = int(group["attempts"].sum())
        accuracy_successes = int(round((group["accuracy"] * group["attempts"]).sum()))
        complete = group[group["eligible_at_10"]]
        pass_successes = int(complete["pass_at_10"].sum())
        majority_successes = int(complete["majority_at_10"].sum())
        block_trials = len(complete)
        accuracy_ci = wilson_interval(accuracy_successes, accuracy_trials)
        pass_ci = wilson_interval(pass_successes, block_trials)
        majority_ci = wilson_interval(majority_successes, block_trials)
        metric_rows.append({
            "mask_protocol": mask_protocol,
            "prompt_type": prompt_type,
            "method": method,
            "layer": int(layer),
            "top_k": int(k),
            "accuracy": accuracy_successes / accuracy_trials,
            "accuracy_ci_low": accuracy_ci[0],
            "accuracy_ci_high": accuracy_ci[1],
            "pass_at_10": pass_successes / block_trials if block_trials else np.nan,
            "pass_at_10_ci_low": pass_ci[0],
            "pass_at_10_ci_high": pass_ci[1],
            "majority_at_10": majority_successes / block_trials if block_trials else np.nan,
            "majority_at_10_ci_low": majority_ci[0],
            "majority_at_10_ci_high": majority_ci[1],
            "individual_responses": accuracy_trials,
            "complete_adapter_blocks_of_10": block_trials,
            "incomplete_blocks_excluded": int((~group["eligible_at_10"]).sum()),
        })
    return pd.DataFrame(metric_rows), units

paper_metrics, paper_units = build_paper_metrics(valid_aggregate)
paper_metrics.to_csv(
    analysis_paths.result_dir / "test_paper_metrics_all_layers.csv", index=False
)
paper_units.to_csv(
    analysis_paths.result_dir / "test_paper_metric_units.csv", index=False
)

paper_layer = analysis_config["paper_reference"]["reported_layer"]
validation_layer = analysis_config["validation_anchor"]["shared_response_layer"]
headline_paper_metrics = paper_metrics[
    paper_metrics["layer"].isin([paper_layer, validation_layer])
    & paper_metrics["mask_protocol"].isin(["global_emitted_ids", "position_actual_token"])
].sort_values(["prompt_type", "layer", "mask_protocol", "top_k", "method"])
headline_paper_metrics.to_csv(
    analysis_paths.result_dir / "test_headline_paper_metrics.csv", index=False
)
display(headline_paper_metrics)


## Plot paper-style metrics over all layers

Solid lines are our primary global emitted-ID mask; dotted lines are the
position-only mask diagnostic. The vertical lines mark paper layer index 32
and validation-frozen Qwen layer 40. Higher is better on every panel.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
metric_labels = {
    "accuracy": "Accuracy: fraction of individual responses",
    "pass_at_10": "Pass@10: at least one hit in each block of 10",
    "majority_at_10": "Majority@10: frequency-voted top-k",
}
method_labels = {"logit_lens": "Logit Lens", "jlens": "J-Lens"}

for prompt_type in ("standard", "direct"):
    for k in analysis_config["readout"]["top_ks"]:
        figure, axes = plt.subplots(1, 3, figsize=(19, 5), sharey=True)
        subset = paper_metrics[
            paper_metrics["prompt_type"].eq(prompt_type)
            & paper_metrics["top_k"].eq(k)
            & paper_metrics["mask_protocol"].isin([
                "global_emitted_ids", "position_actual_token"
            ])
        ]
        for axis, (metric, label) in zip(axes, metric_labels.items()):
            for (method, mask_protocol), group in subset.groupby(
                ["method", "mask_protocol"]
            ):
                group = group.sort_values("layer")
                axis.plot(
                    group["layer"],
                    group[metric],
                    label=(
                        f"{method_labels[method]} | "
                        f"{'global mask (primary)' if mask_protocol == 'global_emitted_ids' else 'position mask'}"
                    ),
                    linestyle="-" if mask_protocol == "global_emitted_ids" else ":",
                    linewidth=2,
                )
            axis.axvline(paper_layer, color="black", linestyle="--", alpha=0.55, label="paper index 32")
            axis.axvline(validation_layer, color="purple", linestyle="--", alpha=0.55, label="validation-frozen layer 40")
            axis.set_title(label)
            axis.set_xlabel("Qwen source layer (0–62)")
            axis.set_ylim(-0.02, 1.02)
        axes[0].set_ylabel("success rate; higher is better")
        handles, labels = axes[-1].get_legend_handles_labels()
        figure.legend(handles, labels, loc="lower center", ncol=3, frameon=True)
        figure.suptitle(
            f"{prompt_type.title()} test prompts — top-{k} paper-style metrics\n"
            "20 adapters × 100 prompts; literal own-secret leaks excluded",
            y=1.04,
        )
        figure.text(
            0.5,
            -0.06,
            "These all-layer test curves are exploratory. Confirmatory anchors were fixed before test: layer 40 and gen 5.",
            ha="center",
        )
        figure.tight_layout(rect=(0, 0.08, 1, 1))
        output = analysis_paths.figure_dir / f"test_{prompt_type}_paper_metrics_top{k}.png"
        output.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(output, dpi=180, bbox_inches="tight")
        plt.show()


## Mask sensitivity at fixed anchors

If LL/J-Lens ordering changes materially between `global_emitted_ids`,
`position_actual_token`, and `unmasked`, the conclusion depends on the masking
definition rather than only on the lens implementation.


In [ ]:
mask_sensitivity = (
    valid_aggregate[
        valid_aggregate["layer"].isin(anchor_layers)
    ]
    .groupby(
        ["prompt_type", "layer", "mask_protocol", "method"], as_index=False
    )
    .agg(
        sequences=("prompt_id", "size"),
        median_rank=("target_rank", "median"),
        geometric_mean_rank=("target_rank", geometric_mean_rank),
        mrr=("target_reciprocal_rank", "mean"),
        hit_at_1=("target_hit_top1", "mean"),
        hit_at_5=("target_hit_top5", "mean"),
        mean_probability_mass=("target_probability_mass", "mean"),
    )
    .sort_values(["prompt_type", "layer", "mask_protocol", "method"])
)
mask_sensitivity.to_csv(
    analysis_paths.result_dir / "test_mask_sensitivity_at_anchors.csv", index=False
)
display(mask_sensitivity)


## Load detailed position rows

This is the large table. We load only analysis columns first; decoded top-10
JSON is loaded later for a small inspection slice. Primary J-Lens summaries
mark absolute positions below 16 as outside its fitting-position domain.


In [ ]:
position_columns = [
    "prompt_id", "prompt_type", "condition", "method", "layer", "position",
    "position_role", "position_label", "position_from_prompt_end",
    "relative_response_position", "assistant_header_offset", "observed_token_id",
    "observed_token", "prediction_target_token", "token_kind", "context",
    "jlens_in_fit_position_domain", "own_secret_leaked", "target_rank",
    "target_reciprocal_rank", "target_log10_rank", "target_rank_percentile",
    "target_hit_top1", "target_hit_top5", "target_hit_top10", "target_hit_top100",
    "target_probability_mass", "target_negative_log_probability",
    "target_logit_margin_to_top1", "target_candidate_rank_20",
    "target_candidate_probability_share", "best_wrong_candidate",
    "position_mask_target_rank", "position_mask_target_reciprocal_rank",
    "unmasked_target_rank", "unmasked_target_reciprocal_rank",
]
positions = pd.concat(
    [pd.read_parquet(path, columns=position_columns) for path in analysis_position_files],
    ignore_index=True,
)
valid_positions = positions[~positions["own_secret_leaked"]].copy()
valid_positions["generated"] = valid_positions["relative_response_position"].notna()
print("position rows:", len(positions), "valid:", len(valid_positions))
print(
    "J-Lens rows outside fit-position domain:",
    int((valid_positions["method"].eq("jlens") & ~valid_positions["jlens_in_fit_position_domain"]).sum()),
)


## Prompt-balanced token-role metrics

Long responses must not dominate. We first average positions inside each
`prompt × adapter × role`, then average those sequence-level values. For J-Lens
the primary table excludes positions below absolute position 16; the raw rows
remain available for exploratory inspection.


In [ ]:
primary_position_domain = valid_positions[
    valid_positions["method"].eq("logit_lens")
    | valid_positions["jlens_in_fit_position_domain"]
].copy()

per_sequence_role = (
    primary_position_domain.groupby(
        ["prompt_type", "condition", "prompt_id", "method", "layer", "position_role"],
        as_index=False,
    )
    .agg(
        token_positions=("position", "size"),
        mean_rr=("target_reciprocal_rank", "mean"),
        median_rank=("target_rank", "median"),
        mean_log10_rank=("target_log10_rank", "mean"),
        hit_at_1=("target_hit_top1", "mean"),
        hit_at_5=("target_hit_top5", "mean"),
        mean_probability_mass=("target_probability_mass", "mean"),
    )
)
role_metrics = (
    per_sequence_role.groupby(
        ["prompt_type", "method", "layer", "position_role"], as_index=False
    )
    .agg(
        prompt_adapter_examples=("prompt_id", "size"),
        mrr=("mean_rr", "mean"),
        median_of_sequence_median_rank=("median_rank", "median"),
        geometric_mean_rank=("mean_log10_rank", lambda s: float(10 ** s.mean())),
        hit_at_1=("hit_at_1", "mean"),
        hit_at_5=("hit_at_5", "mean"),
        mean_probability_mass=("mean_probability_mass", "mean"),
    )
)
role_metrics.to_csv(
    analysis_paths.result_dir / "test_position_role_metrics_by_layer.csv", index=False
)
display(
    role_metrics[
        role_metrics["layer"].isin(anchor_layers)
    ].sort_values(["prompt_type", "layer", "position_role", "method"])
)


## Layer × exact response-offset maps

Each cell is prompt-balanced MRR at an exact generated-token offset. `gen 0`
means the residual after reading the first generated token; offset 0 (not shown
here) is the final prompt separator that predicts the first response token.
Higher/brighter is better. These maps are exploratory on test.


In [ ]:
generated_positions = primary_position_domain[
    primary_position_domain["generated"]
].copy()
per_sequence_exact = (
    generated_positions.groupby(
        ["prompt_type", "condition", "prompt_id", "method", "layer", "position_from_prompt_end"],
        as_index=False,
    )
    .agg(
        reciprocal_rank=("target_reciprocal_rank", "mean"),
        log10_rank=("target_log10_rank", "mean"),
        hit_at_5=("target_hit_top5", "mean"),
    )
)
exact_metrics = (
    per_sequence_exact.groupby(
        ["prompt_type", "method", "layer", "position_from_prompt_end"], as_index=False
    )
    .agg(
        prompt_adapter_examples=("prompt_id", "size"),
        mrr=("reciprocal_rank", "mean"),
        geometric_mean_rank=("log10_rank", lambda s: float(10 ** s.mean())),
        hit_at_5=("hit_at_5", "mean"),
    )
)
exact_metrics.to_csv(
    analysis_paths.result_dir / "test_exact_generated_position_metrics.csv", index=False
)

max_offset_to_plot = 40
for prompt_type in ("standard", "direct"):
    for method in ("logit_lens", "jlens"):
        subset = exact_metrics[
            exact_metrics["prompt_type"].eq(prompt_type)
            & exact_metrics["method"].eq(method)
            & exact_metrics["position_from_prompt_end"].between(1, max_offset_to_plot)
        ]
        pivot = subset.pivot(
            index="layer", columns="position_from_prompt_end", values="mrr"
        ).sort_index(ascending=False)
        figure, axis = plt.subplots(figsize=(18, 10))
        sns.heatmap(
            pivot,
            cmap="viridis",
            vmin=0,
            vmax=min(1.0, float(np.nanquantile(pivot.to_numpy(), 0.99))),
            cbar_kws={"label": "Prompt-balanced MRR (higher is better)"},
            ax=axis,
        )
        axis.set_title(
            f"{method_labels[method]} — {prompt_type} test — generated positions\n"
            "cell = mean reciprocal rank after reading generated token"
        )
        axis.set_xlabel("position from response boundary: 1 = gen 0, 6 = gen 5")
        axis.set_ylabel("Qwen source layer")
        figure.tight_layout()
        output = analysis_paths.figure_dir / f"test_{prompt_type}_{method}_layer_by_generated_position_mrr.png"
        figure.savefig(output, dpi=180, bbox_inches="tight")
        plt.show()


## Confirmatory validation-frozen single-position result

This is the clean test of the choice made before opening test: shared Qwen
layer 40, offset 6 (`gen 5`), global emitted-ID mask. No layer or token is
reselected here.


In [ ]:
frozen_layer = analysis_config["validation_anchor"]["shared_position_layer"]
frozen_offset = analysis_config["validation_anchor"][
    "shared_position_offset_from_response_boundary"
]
frozen_positions = primary_position_domain[
    primary_position_domain["layer"].eq(frozen_layer)
    & primary_position_domain["position_from_prompt_end"].eq(frozen_offset)
].copy()
frozen_summary = (
    frozen_positions.groupby(["prompt_type", "method"], as_index=False)
    .agg(
        sequences=("prompt_id", "size"),
        median_rank=("target_rank", "median"),
        geometric_mean_rank=("target_rank", geometric_mean_rank),
        mrr=("target_reciprocal_rank", "mean"),
        hit_at_1=("target_hit_top1", "mean"),
        hit_at_5=("target_hit_top5", "mean"),
        hit_at_10=("target_hit_top10", "mean"),
        hit_at_100=("target_hit_top100", "mean"),
        median_candidate_rank_20=("target_candidate_rank_20", "median"),
        mean_candidate_probability_share=("target_candidate_probability_share", "mean"),
        mean_probability_mass=("target_probability_mass", "mean"),
        mean_logit_margin_to_top1=("target_logit_margin_to_top1", "mean"),
    )
)
frozen_summary.to_csv(
    analysis_paths.result_dir / "test_frozen_layer40_gen5_metrics.csv", index=False
)
display(frozen_summary)

frozen_by_adapter = (
    frozen_positions.groupby(["prompt_type", "condition", "method"], as_index=False)
    .agg(
        prompts=("prompt_id", "size"),
        median_rank=("target_rank", "median"),
        mrr=("target_reciprocal_rank", "mean"),
        hit_at_1=("target_hit_top1", "mean"),
        hit_at_5=("target_hit_top5", "mean"),
        mean_candidate_share=("target_candidate_probability_share", "mean"),
    )
)
frozen_by_adapter.to_csv(
    analysis_paths.result_dir / "test_frozen_metrics_by_adapter.csv", index=False
)
display(frozen_by_adapter)


## Decoded internal top-token inspection

Notebook 07 saved decoded top-10 tokens for every detailed cell. Here we load
only the validation-frozen layer and a compact set of meaningful positions.
The full per-sequence Parquet files remain sufficient for a future interactive
click-through map.


In [ ]:
inspection_columns = [
    "prompt_id", "prompt_type", "condition", "method", "layer",
    "position", "position_role", "position_label", "position_from_prompt_end",
    "observed_token", "prediction_target_token", "context", "target_word",
    "target_rank", "top1_token", "top10_json", "own_secret_leaked",
]
inspection_parts = []
for path in analysis_position_files:
    part = pd.read_parquet(path, columns=inspection_columns)
    part = part[
        part["layer"].eq(frozen_layer)
        & ~part["own_secret_leaked"]
        & (
            part["position_role"].isin([
                "assistant_turn_start_control",
                "assistant_role_token",
                "assistant_thinking_open_control",
                "assistant_thinking_close_control",
                "response_start_boundary_separator",
            ])
            | part["position_from_prompt_end"].isin([1, 2, 6, 11, 21])
        )
    ]
    if len(part):
        inspection_parts.append(part)
decoded_inspection = pd.concat(inspection_parts, ignore_index=True)
decoded_inspection.to_parquet(
    analysis_paths.result_dir / "test_decoded_top_token_inspection.parquet",
    index=False,
)

def decoded_top_tokens(value):
    return [item["token"] for item in json.loads(value)]

decoded_inspection["decoded_top10"] = decoded_inspection["top10_json"].map(
    decoded_top_tokens
)
with pd.option_context("display.max_colwidth", 120, "display.max_rows", 40):
    display(
        decoded_inspection.sort_values(
            ["prompt_type", "condition", "prompt_id", "method", "position"]
        )[[
            "prompt_id", "condition", "method", "position_label",
            "observed_token", "prediction_target_token", "target_rank", "decoded_top10",
        ]].head(40)
    )


## Frequently dominant internal tokens

This is a descriptive diagnostic, not a secret-detection score. It counts
which decoded token is top-1 most often at the frozen layer for each semantic
position role, after emitted IDs were removed. It can reveal generic template
or punctuation attractors that make vocabulary rank jump between positions.


In [ ]:
top1_frequency = (
    decoded_inspection.groupby(
        ["prompt_type", "method", "position_role", "top1_token"], as_index=False
    )
    .size()
    .rename(columns={"size": "top1_count"})
)
top1_frequency["rank_within_role"] = (
    top1_frequency.groupby(["prompt_type", "method", "position_role"])[
        "top1_count"
    ].rank(method="first", ascending=False)
)
top1_frequency = top1_frequency[
    top1_frequency["rank_within_role"] <= 10
].sort_values(["prompt_type", "method", "position_role", "rank_within_role"])
top1_frequency.to_csv(
    analysis_paths.result_dir / "test_frequent_internal_top1_tokens.csv", index=False
)
display(top1_frequency.head(80))


## Per-adapter heterogeneity and paired LL vs J-Lens comparison

The same `prompt × adapter × layer` appears under both methods, so method
differences are paired. Large spread across words means a pooled headline can
hide weak adapters like the earlier Gold/Blue discrepancy.


In [ ]:
anchor_primary = primary_aggregate[
    primary_aggregate["layer"].isin(anchor_layers)
].copy()
adapter_metrics = (
    anchor_primary.groupby(
        ["prompt_type", "condition", "method", "layer"], as_index=False
    )
    .agg(
        prompts=("prompt_id", "size"),
        median_rank=("target_rank", "median"),
        geometric_mean_rank=("target_rank", geometric_mean_rank),
        mrr=("target_reciprocal_rank", "mean"),
        hit_at_1=("target_hit_top1", "mean"),
        hit_at_5=("target_hit_top5", "mean"),
        mean_candidate_share=("target_candidate_probability_share", "mean"),
    )
)
adapter_metrics.to_csv(
    analysis_paths.result_dir / "test_metrics_by_adapter_at_anchors.csv", index=False
)
display(adapter_metrics)

paired = anchor_primary.pivot_table(
    index=["prompt_type", "condition", "prompt_id", "layer"],
    columns="method",
    values=["target_reciprocal_rank", "target_log10_rank", "target_probability_mass"],
    aggfunc="first",
).reset_index()
paired.columns = [
    "__".join(item).rstrip("__") if isinstance(item, tuple) else item
    for item in paired.columns
]
paired["delta_rr_jlens_minus_ll"] = (
    paired["target_reciprocal_rank__jlens"]
    - paired["target_reciprocal_rank__logit_lens"]
)
paired["delta_log10_rank_jlens_minus_ll"] = (
    paired["target_log10_rank__jlens"]
    - paired["target_log10_rank__logit_lens"]
)
paired_summary = (
    paired.groupby(["prompt_type", "layer"], as_index=False)
    .agg(
        pairs=("prompt_id", "size"),
        mean_delta_rr=("delta_rr_jlens_minus_ll", "mean"),
        median_delta_rr=("delta_rr_jlens_minus_ll", "median"),
        jlens_better_fraction=("delta_rr_jlens_minus_ll", lambda s: float((s > 0).mean())),
        tie_fraction=("delta_rr_jlens_minus_ll", lambda s: float((s == 0).mean())),
        mean_delta_log10_rank=("delta_log10_rank_jlens_minus_ll", "mean"),
    )
)
paired_summary.to_csv(
    analysis_paths.result_dir / "test_paired_jlens_vs_logit_at_anchors.csv", index=False
)
display(paired_summary)


## Cross-candidate specificity across all 20 taboo words

Notebook 07 stored the response-averaged probability of every taboo candidate, not only the adapter's own secret. For every one of the 20 words, this section compares its evidence when the matching adapter is loaded against its evidence under the other 19 adapters.

This is a **closed-set 20-way diagnostic**, not a full-vocabulary rank. Rank 1 means highest among the 20 taboo candidates after the primary global emitted-ID mask. A positive matched-vs-other lift means the word tracks the correct secret rather than merely appearing as a generic or frequent internal guess. Masked non-target candidates are marked unavailable, and literal own-secret leaks remain excluded. Only predeclared anchor layers 32 and 40 are used.


In [ ]:
candidate_words = list(analysis_config["behavior"]["conditions"])
assert len(candidate_words) == 20 and len(set(candidate_words)) == 20
candidate_index = {word: index for index, word in enumerate(candidate_words)}

cross_meta_columns = ["prompt_id", "prompt_type", "condition", "method", "layer"]
cross_meta = anchor_primary[
    cross_meta_columns + ["candidate_probabilities_json", "target_candidate_rank_20"]
].reset_index(drop=True)
assert not cross_meta.duplicated(cross_meta_columns).any()
decoded_candidate_scores = cross_meta["candidate_probabilities_json"].map(json.loads)
assert decoded_candidate_scores.map(lambda scores: set(scores) == set(candidate_words)).all()
candidate_score_matrix = np.asarray([
    [float(scores[word]) for word in candidate_words]
    for scores in decoded_candidate_scores
], dtype=np.float64)
assert np.isfinite(candidate_score_matrix).all()

# Rank matches notebook 07: 1 + number of candidates with a strictly larger score.
# Stable order chooses one confusion label only when top scores tie. A negative
# score is the mask sentinel: that candidate is unavailable for inference.
candidate_order_matrix = np.argsort(-candidate_score_matrix, axis=1, kind="stable")
candidate_rank_matrix = 1 + (
    candidate_score_matrix[:, None, :] > candidate_score_matrix[:, :, None]
).sum(axis=2)
candidate_available_matrix = candidate_score_matrix >= 0
nonnegative_scores = np.clip(candidate_score_matrix, 0.0, None)
score_denominator = nonnegative_scores.sum(axis=1, keepdims=True)
candidate_share_matrix = np.divide(
    nonnegative_scores, score_denominator, out=np.zeros_like(nonnegative_scores),
    where=score_denominator > 0,
)

row_indices = np.arange(len(cross_meta))
target_indices = cross_meta["condition"].map(candidate_index).to_numpy()
saved_target_ranks = cross_meta["target_candidate_rank_20"].astype(int).to_numpy()
np.testing.assert_array_equal(
    candidate_rank_matrix[row_indices, target_indices], saved_target_ranks
)
assert candidate_available_matrix[row_indices, target_indices].all()

predicted_words = np.asarray(candidate_words, dtype=object)[candidate_order_matrix[:, 0]]
cross_predictions = cross_meta[cross_meta_columns].rename(
    columns={"condition": "actual_adapter"}
).copy()
cross_predictions["predicted_candidate_20"] = predicted_words
cross_predictions["top_score_tie_count"] = (
    candidate_score_matrix == candidate_score_matrix.max(axis=1, keepdims=True)
).sum(axis=1)
cross_predictions["correct_candidate_20"] = saved_target_ranks == 1
cross_predictions["true_candidate_rank_20"] = saved_target_ranks
cross_predictions["true_candidate_share_20"] = candidate_share_matrix[row_indices, target_indices]

candidate_count = len(candidate_words)
available_vector = candidate_available_matrix.reshape(-1)
rank_vector = candidate_rank_matrix.reshape(-1)
cross_candidate_detail = pd.DataFrame({
    column: np.repeat(cross_meta[column].to_numpy(), candidate_count)
    for column in cross_meta_columns
}).rename(columns={"condition": "actual_adapter"})
cross_candidate_detail["candidate_word"] = np.tile(candidate_words, len(cross_meta))
cross_candidate_detail["candidate_probability"] = candidate_score_matrix.reshape(-1)
cross_candidate_detail["candidate_probability_share_20"] = candidate_share_matrix.reshape(-1)
cross_candidate_detail["candidate_available"] = available_vector
cross_candidate_detail["candidate_rank_20"] = pd.array(
    np.where(available_vector, rank_vector, np.nan), dtype="Int64"
)
cross_candidate_detail["candidate_reciprocal_rank_20"] = np.where(
    available_vector, 1.0 / rank_vector, np.nan
)
cross_candidate_detail["candidate_is_top1"] = available_vector & (rank_vector == 1)
cross_candidate_detail["is_true_secret"] = (
    cross_candidate_detail["actual_adapter"] == cross_candidate_detail["candidate_word"]
)

cross_predictions.to_csv(
    analysis_paths.result_dir / "test_cross_candidate_predictions_at_anchors.csv", index=False
)
cross_candidate_detail.to_parquet(
    analysis_paths.result_dir / "test_cross_candidate_scores_at_anchors.parquet", index=False
)

cross_candidate_summary = cross_candidate_detail.groupby(
    ["prompt_type", "actual_adapter", "method", "layer", "candidate_word"],
    as_index=False,
).agg(
    prompts=("prompt_id", "size"),
    candidate_available_rate=("candidate_available", "mean"),
    mean_candidate_share_20=("candidate_probability_share_20", "mean"),
    median_candidate_rank_20=("candidate_rank_20", "median"),
    mean_candidate_rr_20=("candidate_reciprocal_rank_20", "mean"),
    candidate_top1_rate=("candidate_is_top1", "mean"),
)
cross_candidate_summary.to_csv(
    analysis_paths.result_dir / "test_cross_candidate_summary_at_anchors.csv", index=False
)

# Core specificity result: each candidate under its matching adapter versus
# the same candidate under all 19 nonmatching adapters.
specificity_rows = []
for keys, part in cross_candidate_detail.groupby(
    ["prompt_type", "method", "layer", "candidate_word"], sort=False
):
    prompt_type, method, layer, candidate_word = keys
    matched = part[part["is_true_secret"]]
    other = part[~part["is_true_secret"]]
    matched_share = float(matched["candidate_probability_share_20"].mean())
    other_share = float(other["candidate_probability_share_20"].mean())
    matched_rank = float(matched["candidate_rank_20"].median())
    other_rank = float(other["candidate_rank_20"].median())
    matched_rr = float(matched["candidate_reciprocal_rank_20"].mean())
    other_rr = float(other["candidate_reciprocal_rank_20"].mean())
    matched_top1 = float(matched["candidate_is_top1"].mean())
    other_top1 = float(other["candidate_is_top1"].mean())
    specificity_rows.append({
        "prompt_type": prompt_type, "method": method, "layer": int(layer),
        "candidate_word": candidate_word, "matched_examples": len(matched),
        "other_adapter_examples": len(other),
        "matched_available_rate": float(matched["candidate_available"].mean()),
        "other_available_rate": float(other["candidate_available"].mean()),
        "matched_mean_share_20": matched_share, "other_mean_share_20": other_share,
        "specificity_share_delta": matched_share - other_share,
        "specificity_share_ratio": matched_share / other_share if other_share > 0 else np.nan,
        "matched_median_rank_20": matched_rank, "other_median_rank_20": other_rank,
        "specificity_median_rank_improvement": other_rank - matched_rank,
        "matched_mean_rr_20": matched_rr, "other_mean_rr_20": other_rr,
        "specificity_rr_delta": matched_rr - other_rr,
        "matched_top1_rate": matched_top1, "other_top1_rate": other_top1,
        "specificity_top1_delta": matched_top1 - other_top1,
    })
cross_candidate_specificity = pd.DataFrame(specificity_rows)
cross_candidate_specificity.to_csv(
    analysis_paths.result_dir / "test_cross_candidate_specificity_at_anchors.csv", index=False
)

cross_candidate_accuracy = cross_predictions.groupby(
    ["prompt_type", "method", "layer"], as_index=False
).agg(
    prompt_adapter_examples=("prompt_id", "size"),
    closed_set_accuracy_20=("correct_candidate_20", "mean"),
    median_true_candidate_rank_20=("true_candidate_rank_20", "median"),
    mean_true_candidate_share_20=("true_candidate_share_20", "mean"),
)
cross_candidate_accuracy.to_csv(
    analysis_paths.result_dir / "test_cross_candidate_accuracy_at_anchors.csv", index=False
)

cross_candidate_confusion = cross_predictions.groupby(
    ["prompt_type", "method", "layer", "actual_adapter", "predicted_candidate_20"],
    as_index=False,
).size().rename(columns={"size": "prompts"})
cross_candidate_confusion["prediction_rate"] = (
    cross_candidate_confusion["prompts"]
    / cross_candidate_confusion.groupby(
        ["prompt_type", "method", "layer", "actual_adapter"]
    )["prompts"].transform("sum")
)
cross_candidate_confusion.to_csv(
    analysis_paths.result_dir / "test_cross_candidate_confusion_at_anchors.csv", index=False
)

display(cross_candidate_accuracy.sort_values(["layer", "prompt_type", "method"]))
with pd.option_context("display.max_rows", 100, "display.max_columns", None):
    display(cross_candidate_specificity.sort_values(
        ["layer", "prompt_type", "method", "specificity_share_delta"],
        ascending=[True, True, True, False],
    ))

for layer in anchor_layers:
    figure, axes = plt.subplots(2, 2, figsize=(24, 20), constrained_layout=True)
    for row_index, prompt_type in enumerate(("standard", "direct")):
        for column_index, method in enumerate(("logit_lens", "jlens")):
            axis = axes[row_index, column_index]
            subset = cross_candidate_confusion[
                cross_candidate_confusion["layer"].eq(layer)
                & cross_candidate_confusion["prompt_type"].eq(prompt_type)
                & cross_candidate_confusion["method"].eq(method)
            ]
            pivot = subset.pivot(
                index="actual_adapter", columns="predicted_candidate_20", values="prediction_rate"
            ).reindex(index=candidate_words, columns=candidate_words).fillna(0.0)
            sns.heatmap(
                pivot, cmap="Blues", vmin=0, vmax=1, square=True,
                cbar_kws={"label": "Fraction of prompts predicted as column word"}, ax=axis,
            )
            for diagonal_index, word in enumerate(candidate_words):
                axis.text(
                    diagonal_index + 0.5, diagonal_index + 0.5, f"{pivot.loc[word, word]:.2f}",
                    ha="center", va="center", fontsize=7, fontweight="bold", color="black",
                )
            axis.set_title(f"{method_labels[method]} — {prompt_type}")
            axis.set_xlabel("highest-scoring candidate among 20")
            axis.set_ylabel("actual loaded adapter")
            axis.tick_params(axis="x", labelrotation=60, labelsize=8)
            axis.tick_params(axis="y", labelrotation=0, labelsize=8)
    figure.suptitle(
        f"20-way taboo-word confusion at Qwen layer {layer}\n"
        "global emitted-ID mask; diagonal labels are closed-set accuracy per adapter", fontsize=17,
    )
    figure.text(
        0.5, -0.01,
        "Rows = actual adapter; columns = top candidate. Dark off-diagonal cells show systematic confusion. Literal own-secret leaks are excluded.",
        ha="center", fontsize=11,
    )
    output = analysis_paths.figure_dir / f"test_cross_candidate_confusion_layer_{layer}.png"
    figure.savefig(output, dpi=180, bbox_inches="tight")
    plt.show()


## Paired adapter-versus-base specificity control

The previous section compares each word's matching adapter with the other 19 adapters. This section adds the stricter control: the same word under its matching LoRA versus the base model on the **same prompt, method, source layer, and mask protocol**.

For every candidate, positive deltas in 20-way probability share and reciprocal rank mean that loading the matching adapter strengthens that candidate beyond its base-model prior. Full-vocabulary rank provides the corresponding open-vocabulary comparison. Base candidates emitted in the base response are unavailable under the global emitted-ID mask; availability is reported explicitly, and the unmasked protocol remains a diagnostic. Anchor-level 95% intervals bootstrap prompts as clusters because 20 adapters share each prompt. All-layer summaries are descriptive; layers 32 and 40 are the predeclared anchors.


In [ ]:
base_aggregate = base_aggregate.reset_index(drop=True)
base_pair_keys = ["prompt_id", "prompt_type", "method", "layer", "mask_protocol"]
assert not base_aggregate.duplicated(base_pair_keys).any()

def candidate_json_matrix(frame, column):
    decoded = frame[column].map(json.loads)
    assert decoded.map(lambda values: set(values) == set(candidate_words)).all()
    return np.asarray([
        [np.nan if values[word] is None else float(values[word]) for word in candidate_words]
        for values in decoded
    ], dtype=np.float64)

base_candidate_probability_matrix = candidate_json_matrix(
    base_aggregate, "candidate_probabilities_json"
)
base_candidate_mass_matrix = candidate_json_matrix(
    base_aggregate, "candidate_probability_masses_json"
)
base_candidate_rank20_matrix = candidate_json_matrix(
    base_aggregate, "candidate_ranks_20_json"
)
base_candidate_share20_matrix = candidate_json_matrix(
    base_aggregate, "candidate_probability_shares_20_json"
)
base_candidate_full_rank_matrix = candidate_json_matrix(
    base_aggregate, "candidate_full_vocab_ranks_json"
)
base_candidate_available_matrix = np.isfinite(base_candidate_rank20_matrix)

adapter_base_summaries = []
adapter_base_anchor_pairs = []
for candidate_index, candidate_word in enumerate(candidate_words):
    base_word = base_aggregate[base_pair_keys].copy()
    base_word["candidate_word"] = candidate_word
    base_word["base_candidate_available"] = base_candidate_available_matrix[:, candidate_index]
    base_word["base_candidate_probability"] = base_candidate_probability_matrix[:, candidate_index]
    base_word["base_candidate_probability_mass"] = base_candidate_mass_matrix[:, candidate_index]
    base_word["base_candidate_rank_20"] = base_candidate_rank20_matrix[:, candidate_index]
    base_word["base_candidate_share_20"] = base_candidate_share20_matrix[:, candidate_index]
    base_word["base_full_vocab_rank"] = base_candidate_full_rank_matrix[:, candidate_index]
    base_word["base_full_vocab_rr"] = 1.0 / base_word["base_full_vocab_rank"]

    adapter_word = valid_aggregate[
        valid_aggregate["condition"].eq(candidate_word)
    ][base_pair_keys + [
        "target_probability", "target_probability_mass", "target_rank",
        "target_reciprocal_rank", "target_candidate_rank_20",
        "target_candidate_probability_share",
    ]].copy()
    assert not adapter_word.duplicated(base_pair_keys).any()
    adapter_word["candidate_word"] = candidate_word
    adapter_word = adapter_word.rename(columns={
        "target_probability": "adapter_candidate_probability",
        "target_probability_mass": "adapter_candidate_probability_mass",
        "target_rank": "adapter_full_vocab_rank",
        "target_reciprocal_rank": "adapter_full_vocab_rr",
        "target_candidate_rank_20": "adapter_candidate_rank_20",
        "target_candidate_probability_share": "adapter_candidate_share_20",
    })
    paired = adapter_word.merge(
        base_word, on=base_pair_keys + ["candidate_word"],
        how="inner", validate="one_to_one",
    )
    assert len(paired) == len(adapter_word)
    paired["paired_candidate_available"] = (
        paired["base_candidate_available"]
        & paired["adapter_full_vocab_rank"].notna()
    )
    available = paired["paired_candidate_available"]
    paired["adapter_candidate_rr_20"] = 1.0 / paired["adapter_candidate_rank_20"]
    paired["base_candidate_rr_20"] = 1.0 / paired["base_candidate_rank_20"]
    paired["delta_candidate_share_20"] = np.where(
        available, paired["adapter_candidate_share_20"] - paired["base_candidate_share_20"], np.nan
    )
    paired["delta_candidate_rr_20"] = np.where(
        available, paired["adapter_candidate_rr_20"] - paired["base_candidate_rr_20"], np.nan
    )
    paired["delta_full_vocab_rr"] = np.where(
        available, paired["adapter_full_vocab_rr"] - paired["base_full_vocab_rr"], np.nan
    )
    paired["adapter_share_beats_base"] = np.where(
        available, paired["adapter_candidate_share_20"] > paired["base_candidate_share_20"], np.nan
    )
    paired["adapter_full_rank_beats_base"] = np.where(
        available, paired["adapter_full_vocab_rank"] < paired["base_full_vocab_rank"], np.nan
    )
    paired["adapter_hit_at_5"] = np.where(
        available, paired["adapter_full_vocab_rank"] <= 5, np.nan
    )
    paired["base_hit_at_5"] = np.where(
        available, paired["base_full_vocab_rank"] <= 5, np.nan
    )

    per_candidate = paired.groupby(
        ["prompt_type", "method", "layer", "mask_protocol"], as_index=False
    ).agg(
        paired_examples=("prompt_id", "size"),
        available_pairs=("paired_candidate_available", "sum"),
        base_candidate_available_rate=("base_candidate_available", "mean"),
        mean_delta_candidate_share_20=("delta_candidate_share_20", "mean"),
        mean_delta_candidate_rr_20=("delta_candidate_rr_20", "mean"),
        mean_delta_full_vocab_rr=("delta_full_vocab_rr", "mean"),
        adapter_share_win_rate=("adapter_share_beats_base", "mean"),
        adapter_full_rank_win_rate=("adapter_full_rank_beats_base", "mean"),
        median_adapter_full_vocab_rank=("adapter_full_vocab_rank", "median"),
        median_base_full_vocab_rank=("base_full_vocab_rank", "median"),
        adapter_hit_at_5=("adapter_hit_at_5", "mean"),
        base_hit_at_5=("base_hit_at_5", "mean"),
    )
    per_candidate["candidate_word"] = candidate_word
    adapter_base_summaries.append(per_candidate)
    adapter_base_anchor_pairs.append(paired[paired["layer"].isin(anchor_layers)])

adapter_vs_base_by_candidate_layer = pd.concat(adapter_base_summaries, ignore_index=True)
adapter_vs_base_anchor_pairs = pd.concat(adapter_base_anchor_pairs, ignore_index=True)
adapter_vs_base_by_candidate_layer.to_csv(
    analysis_paths.result_dir / "test_adapter_vs_base_by_candidate_all_layers.csv", index=False
)
adapter_vs_base_anchor_pairs.to_parquet(
    analysis_paths.result_dir / "test_adapter_vs_base_paired_at_anchors.parquet", index=False
)

def prompt_cluster_bootstrap_mean_interval(frame, value_column, seed, draws=2000):
    # Twenty adapters share each prompt, so resample prompt-level means rather
    # than pretending all prompt-adapter rows are independent observations.
    clean = frame.groupby("prompt_id")[value_column].mean().dropna().to_numpy(dtype=np.float64)
    clean = clean[np.isfinite(clean)]
    if len(clean) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    sampled_means = rng.choice(clean, size=(draws, len(clean)), replace=True).mean(axis=1)
    return tuple(np.quantile(sampled_means, [0.025, 0.975]))

pooled_rows = []
for group_index, (keys, part) in enumerate(adapter_vs_base_anchor_pairs.groupby(
    ["prompt_type", "method", "layer", "mask_protocol"], sort=True
)):
    prompt_type, method, layer, mask_protocol = keys
    usable = part[part["paired_candidate_available"]]
    share_low, share_high = prompt_cluster_bootstrap_mean_interval(
        usable, "delta_candidate_share_20", seed=17000 + group_index
    )
    full_rr_low, full_rr_high = prompt_cluster_bootstrap_mean_interval(
        usable, "delta_full_vocab_rr", seed=27000 + group_index
    )
    pooled_rows.append({
        "prompt_type": prompt_type, "method": method, "layer": int(layer),
        "mask_protocol": mask_protocol, "paired_examples": len(part),
        "available_pairs": len(usable),
        "base_candidate_available_rate": float(part["base_candidate_available"].mean()),
        "mean_delta_candidate_share_20": float(usable["delta_candidate_share_20"].mean()),
        "mean_delta_candidate_share_20_ci_low": float(share_low),
        "mean_delta_candidate_share_20_ci_high": float(share_high),
        "adapter_share_win_rate": float(usable["adapter_share_beats_base"].mean()),
        "mean_delta_candidate_rr_20": float(usable["delta_candidate_rr_20"].mean()),
        "mean_delta_full_vocab_rr": float(usable["delta_full_vocab_rr"].mean()),
        "mean_delta_full_vocab_rr_ci_low": float(full_rr_low),
        "mean_delta_full_vocab_rr_ci_high": float(full_rr_high),
        "adapter_full_rank_win_rate": float(usable["adapter_full_rank_beats_base"].mean()),
        "median_adapter_full_vocab_rank": float(usable["adapter_full_vocab_rank"].median()),
        "median_base_full_vocab_rank": float(usable["base_full_vocab_rank"].median()),
        "adapter_hit_at_5": float(usable["adapter_hit_at_5"].mean()),
        "base_hit_at_5": float(usable["base_hit_at_5"].mean()),
    })
adapter_vs_base_anchor_summary = pd.DataFrame(pooled_rows)
adapter_vs_base_anchor_summary.to_csv(
    analysis_paths.result_dir / "test_adapter_vs_base_summary_at_anchors.csv", index=False
)
display(adapter_vs_base_anchor_summary.sort_values(
    ["layer", "mask_protocol", "prompt_type", "method"]
))

primary_base_comparison = adapter_vs_base_by_candidate_layer[
    adapter_vs_base_by_candidate_layer["mask_protocol"].eq(
        analysis_config["readout"]["primary_mask_protocol"]
    )
].copy()
for layer in anchor_layers:
    figure, axes = plt.subplots(2, 2, figsize=(22, 13), constrained_layout=True)
    for row_index, prompt_type in enumerate(("standard", "direct")):
        for column_index, (metric, label) in enumerate((
            ("mean_delta_candidate_share_20", "LoRA − base: mean 20-way probability share"),
            ("mean_delta_full_vocab_rr", "LoRA − base: mean full-vocabulary reciprocal rank"),
        )):
            axis = axes[row_index, column_index]
            subset = primary_base_comparison[
                primary_base_comparison["layer"].eq(layer)
                & primary_base_comparison["prompt_type"].eq(prompt_type)
            ]
            x = np.arange(len(candidate_words))
            for method_index, method in enumerate(("logit_lens", "jlens")):
                method_part = subset[subset["method"].eq(method)].set_index("candidate_word").reindex(candidate_words)
                offset = -0.13 if method_index == 0 else 0.13
                axis.scatter(
                    x + offset, method_part[metric], s=42,
                    label=method_labels[method], alpha=0.85,
                )
            axis.axhline(0, color="black", linewidth=1, linestyle="--")
            axis.set_xticks(x, candidate_words, rotation=60, ha="right")
            axis.set_ylabel(label + " (positive favors LoRA)")
            axis.set_title(prompt_type)
            axis.grid(axis="y", alpha=0.25)
            axis.legend(title="Readout")
    figure.suptitle(
        f"Matching-secret adapter versus base control at Qwen layer {layer}\n"
        "same prompt and global emitted-ID mask; each point is one candidate word",
        fontsize=16,
    )
    figure.text(
        0.5, -0.015,
        "Consistently positive values across words support adapter-specific secret evidence rather than a base-model word prior.",
        ha="center", fontsize=11,
    )
    output = analysis_paths.figure_dir / f"test_adapter_vs_base_specificity_layer_{layer}.png"
    figure.savefig(output, dpi=180, bbox_inches="tight")
    plt.show()


### Base-model decoded-token inspection

The quantitative comparison above uses response averages. This compact audit also exposes base-model decoded top-10 tokens at the frozen layer for the same meaningful prompt/header and generated-response offsets used in the adapter inspection. It is diagnostic rather than a headline metric.


In [ ]:
base_inspection_columns = [
    "prompt_id", "prompt_type", "condition", "method", "layer",
    "position", "position_role", "position_label", "position_from_prompt_end",
    "observed_token", "prediction_target_token", "context",
    "candidate_probabilities_json", "candidate_ranks_20_json",
    "top1_token", "top10_json",
]
base_inspection_parts = []
for path in base_analysis_position_files:
    part = pd.read_parquet(path, columns=base_inspection_columns)
    part = part[
        part["layer"].eq(frozen_layer)
        & (
            part["position_role"].isin([
                "assistant_turn_start_control", "assistant_role_token",
                "assistant_thinking_open_control", "assistant_thinking_close_control",
                "response_start_boundary_separator",
            ])
            | part["position_from_prompt_end"].isin([1, 2, 6, 11, 21])
        )
    ]
    if len(part):
        base_inspection_parts.append(part)
base_decoded_inspection = pd.concat(base_inspection_parts, ignore_index=True)
base_decoded_inspection["decoded_top10"] = base_decoded_inspection["top10_json"].map(
    decoded_top_tokens
)
base_decoded_inspection.to_parquet(
    analysis_paths.result_dir / "test_base_decoded_top_token_inspection.parquet",
    index=False,
)
with pd.option_context("display.max_colwidth", 120, "display.max_rows", 40):
    display(base_decoded_inspection.sort_values(
        ["prompt_type", "prompt_id", "method", "position"]
    )[[
        "prompt_id", "method", "position_label", "observed_token",
        "prediction_target_token", "top1_token", "decoded_top10",
    ]].head(40))


## Exploratory best test layers — never use these as confirmatory selection

This ranks layers only to describe the test map. The valid confirmatory answer
remains layer 40 / gen 5 frozen on validation.


In [ ]:
exploratory_best_layers = (
    rank_by_layer.sort_values(
        ["prompt_type", "method", "mrr", "hit_at_5", "layer"],
        ascending=[True, True, False, False, True],
    )
    .groupby(["prompt_type", "method"], as_index=False)
    .head(5)
)
exploratory_best_layers.to_csv(
    analysis_paths.result_dir / "test_exploratory_best_layers.csv", index=False
)
display(exploratory_best_layers)


## Machine-readable analysis completion

The JSON records which results are confirmatory and which are exploratory, so
later reporting cannot accidentally present a test-selected optimum as held-out
evidence.


In [ ]:
analysis_completion = {
    "schema_version": 2,
    "created_utc": utc_now(),
    "run_id": ANALYSIS_RUN_ID,
    "base_control_run_id": BASE_ANALYSIS_RUN_ID,
    "literal_own_secret_leaks_excluded": len(leak_keys),
    "primary_mask_protocol": analysis_config["readout"]["primary_mask_protocol"],
    "confirmatory_response_average_layer": validation_layer,
    "confirmatory_single_position": {
        "layer": frozen_layer,
        "position_from_prompt_end": frozen_offset,
        "label": analysis_config["validation_anchor"]["shared_position_label"],
    },
    "paper_numeric_layer_index": paper_layer,
    "all_layer_and_position_scans_are_exploratory": True,
    "position_rows_analyzed": len(valid_positions),
    "aggregate_rows_analyzed": len(valid_aggregate),
    "base_aggregate_rows_analyzed": len(base_aggregate),
    "artifacts": {
        "paper_metrics": "test_paper_metrics_all_layers.csv",
        "rank_metrics": "test_rank_probability_metrics_by_layer.csv",
        "frozen_metrics": "test_frozen_layer40_gen5_metrics.csv",
        "decoded_inspection": "test_decoded_top_token_inspection.parquet",
        "cross_candidate_predictions": "test_cross_candidate_predictions_at_anchors.csv",
        "cross_candidate_scores": "test_cross_candidate_scores_at_anchors.parquet",
        "cross_candidate_summary": "test_cross_candidate_summary_at_anchors.csv",
        "cross_candidate_specificity": "test_cross_candidate_specificity_at_anchors.csv",
        "cross_candidate_accuracy": "test_cross_candidate_accuracy_at_anchors.csv",
        "cross_candidate_confusion": "test_cross_candidate_confusion_at_anchors.csv",
        "adapter_vs_base_all_layers": "test_adapter_vs_base_by_candidate_all_layers.csv",
        "adapter_vs_base_anchor_pairs": "test_adapter_vs_base_paired_at_anchors.parquet",
        "adapter_vs_base_anchor_summary": "test_adapter_vs_base_summary_at_anchors.csv",
        "base_decoded_inspection": "test_base_decoded_top_token_inspection.parquet",
    },
}
(analysis_paths.result_dir / "test_analysis_completion.json").write_text(
    json.dumps(analysis_completion, indent=2), encoding="utf-8"
)
display(analysis_completion)
